In [14]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import dgl
import dgl.nn as dglnn

DEVICE = torch.device("cpu")

def seed_all(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

seed_all(42)

DATA = Path("../data/amazon_computers.pt")
OUT = Path("../outputs/")
OUT.mkdir(parents=True, exist_ok=True)


In [15]:
ck = torch.load(DATA, weights_only=False)
g, feats, labels = ck["graph"], ck["feats"], ck["labels"]
idx_tr, idx_va, idx_te = ck["idx_train"], ck["idx_val"], ck["idx_test"]

g = g.to(DEVICE)
feats = feats.to(DEVICE)
labels = labels.to(DEVICE)

N, d = feats.shape
C = int(labels.max() + 1)
print(f"N={N} E={g.num_edges()} feat={d} C={C}")
print(f"train {len(idx_tr)} val {len(idx_va)} test {len(idx_te)}")


N=13752 E=505474 feat=767 C=10
train 200 val 300 test 13252


In [16]:
def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = dglnn.GraphConv(d, 512, activation=F.relu)
        self.c2 = dglnn.GraphConv(512, C)
        self.drop = nn.Dropout(0.5)

    def forward(self, g, x):
        h = self.drop(self.c1(g, x))
        return self.c2(g, h)

model = GCN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=5e-4)
print(model)


GCN(
  (c1): GraphConv(in=767, out=512, normalization=both, activation=<function relu at 0x7faaa3f73ce0>)
  (c2): GraphConv(in=512, out=10, normalization=both, activation=None)
  (drop): Dropout(p=0.5, inplace=False)
)


In [17]:
best_val = 0
best_state = None
wait = 0
best_test = 0

for epoch in range(1, 501):
    model.train()
    opt.zero_grad()
    logits = model(g, feats)
    loss = F.cross_entropy(logits[idx_tr], labels[idx_tr])
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        lg = model(g, feats)
        v = accuracy(lg[idx_va], labels[idx_va])
        te = accuracy(lg[idx_te], labels[idx_te])

    if v > best_val:
        best_val = v
        best_test = te
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1

    if epoch % 40 == 0:
        print(f"{epoch} loss{loss.item():.3f} va{v:.4f} te{te:.4f} best{best_val:.4f}/{best_test:.4f}")

    if wait >= 50:
        print(f"early stop {epoch}")
        break


40 loss0.824 va0.8467 te0.7868 best0.8567/0.8062
80 loss0.198 va0.8733 te0.8238 best0.8833/0.8215
120 loss0.139 va0.8767 te0.8368 best0.9000/0.8232
early stop 155


In [18]:
model.load_state_dict(best_state)

with torch.no_grad():
    logits = model(g, feats)
    test_acc = accuracy(logits[idx_te], labels[idx_te])
    print(f"teacher {test_acc:.4f} val {best_val:.4f}")

torch.save({"state": best_state}, OUT / "teacher_gcn.pt")
np.savez_compressed(OUT / "teacher_logits.npz", logits=logits.detach().cpu().numpy())


teacher 0.8232 val 0.9000
